# NBA 3.1 - 10 Kerendia Prior Writing (all claim types)

Builds `KERENDIA_NBRX_IN_LAST_{1..6}_MONTH` and `KERENDIA_TRX_IN_LAST_{1..6}_MONTH` per HCP and claims cohort, from raw IQVIA LAAD `STG_CAR_IQV_LAAD_FCT_RX`, counting **all claim types (PD, RJ, RV)**. `26_Features_NBRx` counts paid claims only (LRx V6).

The claim pull and the NBRx logic are copied from `Kerendia_NBRX_From_Raw_FCT_RX` (`01_HCP_Universe/Raw Claims NBRX Calculation/`):
* **NBRx:** the patient's first-ever Kerendia claim, or a claim more than 365 days after the patient's previous Kerendia claim. Only one claim per patient per day counts (lowest `CLAIM_ID`).
* **TRx:** every Kerendia claim counts as 1.

The NBRx flag is computed on the patient's **full** claim history, across all HCPs, before any date or universe filter. This keeps "first-ever" correct.

| Item | Value |
| --- | --- |
| Cohorts | Claims cohorts: `kerendia_nba_3_1_volume/cohorts/claims/cohorts.parquet` |
| HCP universe | `kerendia_nba_3_1_volume/HCP_Universe/` |
| Window | N calendar months ending with the cohort's `lookback_end_date` (N = 1 to 6), same as `step3_lag` in 3.0 |
| Monthly table | `kerendia_nba_3_1_volume/hcp_level_model/ops_pipeline/query_output/kerendia_claims_monthly_from_raw_fct_rx` |
| Output | `kerendia_nba_3_1_volume/notebook_output/hcp_features/kerendia_prior_writing` |

Output grain: one row per 3.1 universe HCP x claims cohort (`HCP_COHORT_ID = PROVIDER_ID_cohort_id`). HCPs without Kerendia claims in a window get 0.

Run on a classic cluster: Snowflake is needed for the pull.

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
import os, json, builtins   # builtins: 00_connection does `from pyspark.sql.functions import *` (shadows min/max/sum/round)
from datetime import datetime as dt
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.window import Window

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
COHORTS_DIR = f"{VOL31}/cohorts/claims"                      # claims feature family
HCP_UNIVERSE_PATH = f"{VOL31}/HCP_Universe"
MONTHLY_PATH = f"{VOL31}/hcp_level_model/ops_pipeline/query_output/kerendia_claims_monthly_from_raw_fct_rx"
HCP_FEATURES_OUT_31 = f"{VOL31}/notebook_output/hcp_features"
OUTPUT_PATH = f"{HCP_FEATURES_OUT_31}/kerendia_prior_writing"

# Saved HCP-level table from Kerendia_NBRX_From_Raw_FCT_RX (lifetime totals) - used only as a cross-check
HCP_LEVEL_REF_PATH = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/"
                      "hcp_level_model/dev_folder_3_1/notebook_output/kerendia_nbrx_from_raw_fct_rx")
LRX_TABLE = "`ph-com-ai-us-prod-catalog-2553213839599380`.`kerendia_laad_based_lrx`.KERENDIA_LAAD_BASED_LRX_V6"

WINDOWS = [1, 2, 3, 4, 5, 6]      # N calendar months ending with lookback_end_date (same as 3.0 step3_lag)

# Claim-type groups (CLAIM_TYPE in raw FCT_RX: PD = paid, RJ = rejected, RV = reversed)
CLAIM_GROUPS = {
    "OVERALL":  F.lit(True),                                # PD + RJ + RV
    "PD":       F.col("CLAIM_TYPE") == "PD",                # paid (check column, not a feature)
    "RJ":       F.col("CLAIM_TYPE") == "RJ",                # rejected
    "RV":       F.col("CLAIM_TYPE") == "RV",                # reversed
    "APPROVED": F.col("CLAIM_TYPE").isin("PD", "RV"),       # approved = paid + reversed
}
# Feature groups: monthly-table suffix -> feature-name infix ('' keeps the 3.0 name for all claim types)
FEATURE_GROUPS = {"OVERALL": "", "RJ": "_REJECTED", "RV": "_REVERSED", "APPROVED": "_APPROVED"}
REFRESH_PULL = True              # False = reuse the saved monthly table

In [0]:
cohorts = pd.read_parquet(f"{COHORTS_DIR}/cohorts.parquet")
cohorts = cohorts.sort_values("cohort_id", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)
with open(f"{COHORTS_DIR}/metadata.json") as f:
    meta = json.load(f)
assert meta["feature_family"] == "claims", meta["feature_family"]
assert len(cohorts) == meta["cohort_count"] and (cohorts["offset_days"] == 0).all()
lookback_end_max = pd.to_datetime(cohorts["lookback_end_date"]).max().date()

hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID").distinct()
n_hcp = hcp_universe.count()

coh = (spark.createDataFrame(cohorts[["cohort_id", "lookback_end_date"]].astype(str))
       .withColumn("LOOKBACK_END_M", F.to_date("lookback_end_date")).drop("lookback_end_date"))

print(f"Claims cohorts: {len(cohorts)} ({cohorts.cohort_id.iloc[0]}..{cohorts.cohort_id.iloc[-1]}), latest lookback end {lookback_end_max}")
print(f"HCP universe: {n_hcp:,}")
display(cohorts)

In [0]:
if REFRESH_PULL:
    # Steps 1-3: copied from Kerendia_NBRX_From_Raw_FCT_RX (all claim types: PD, RJ, RV; full history)
    query_kerendia = '''
        SELECT DISTINCT
            A.CLAIM_ID,
            A.PATIENT_ID,
            TO_DATE(A.SVC_DT) AS SVC_DT,
            A.PROVIDER_ID,
            A.CLAIM_TYPE,
            TRIM(UPPER(B.PRODUCT_GROUP)) AS PRODUCT_GROUP
        FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX A
        INNER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD B
        USING (NDC_CD)
        WHERE TRIM(UPPER(B.PRODUCT_GROUP)) = 'KERENDIA'
    '''
    kerendia_claims = Get_Data_Snowflakes(query_kerendia).withColumn("PROVIDER_ID", F.col("PROVIDER_ID").cast("string"))

    # Step 2: one claim per patient per day (lowest CLAIM_ID)
    dedup_window = Window.partitionBy("PATIENT_ID", "PRODUCT_GROUP", "SVC_DT").orderBy("CLAIM_ID")
    # Step 3: NBRx = first-ever claim, or > 365 days after the previous claim (patient history across all HCPs)
    window_spec = Window.partitionBy("PATIENT_ID", "PRODUCT_GROUP").orderBy("SVC_DT")
    kc = (kerendia_claims
          .withColumn("CLAIM_RANK", F.row_number().over(dedup_window))
          .withColumn("MIN_SVC_DT", F.min("SVC_DT").over(window_spec))
          .withColumn("LAG_SVC_DT", F.lag("SVC_DT").over(window_spec)))
    kc = kc.withColumn(
        "NBRX_FLAG",
        F.when((F.col("MIN_SVC_DT") == F.col("SVC_DT")) & (F.col("CLAIM_RANK") == 1), 1)
         .when(F.col("LAG_SVC_DT").isNotNull() & (F.col("LAG_SVC_DT") < F.expr("date_add(SVC_DT, -365)")) & (F.col("CLAIM_RANK") == 1), 1)
         .otherwise(0))

    # Claim-type check: the splits below assume CLAIM_TYPE is only PD / RJ / RV
    kc = kc.withColumn("CLAIM_TYPE", F.trim(F.upper(F.col("CLAIM_TYPE"))))
    print("Claim type breakdown (claims, NBRx):")
    kc.groupBy("CLAIM_TYPE").agg(F.count("*").alias("claims"), F.sum("NBRX_FLAG").alias("nbrx")).orderBy(F.desc("claims")).show()

    # Step 4: monthly totals per HCP and claim-type group (replaces the HCP-level lifetime rollup of the source notebook)
    #   OVERALL = PD + RJ + RV | PD = paid | RJ = rejected | RV = reversed | APPROVED = PD + RV
    aggs_m = []
    for grp, cond in CLAIM_GROUPS.items():
        aggs_m.append(F.sum(F.when((F.col("NBRX_FLAG") == 1) & cond, 1).otherwise(0)).alias(f"NBRX_KERENDIA_{grp}"))
        aggs_m.append(F.sum(F.when(cond, 1).otherwise(0)).alias(f"TRX_KERENDIA_{grp}"))
    monthly_out = (kc.filter(F.col("PROVIDER_ID").isNotNull())
        .withColumn("SVC_DT_M", F.trunc("SVC_DT", "month"))
        .groupBy("PROVIDER_ID", "SVC_DT_M")
        .agg(*aggs_m)
        .withColumn("PULL_DT", F.current_date()))
    (monthly_out.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(MONTHLY_PATH))
    print(f"Saved monthly table -> {MONTHLY_PATH}")

monthly_all = spark.read.format("delta").load(MONTHLY_PATH)
summary = monthly_all.agg(F.countDistinct("PROVIDER_ID").alias("hcps"), F.min("SVC_DT_M").alias("first_month"), F.max("SVC_DT_M").alias("last_month"),
                          *[F.sum(f"{m}_KERENDIA_{g}").alias(f"{m.lower()}_{g.lower()}") for m in ("NBRX", "TRX") for g in CLAIM_GROUPS])
display(summary)
# Claim-type groups must add up: OVERALL = PD + RJ + RV, APPROVED = PD + RV
for m in ("NBRX", "TRX"):
    bad_sum = monthly_all.filter(f"{m}_KERENDIA_OVERALL != {m}_KERENDIA_PD + {m}_KERENDIA_RJ + {m}_KERENDIA_RV").count()
    bad_app = monthly_all.filter(f"{m}_KERENDIA_APPROVED != {m}_KERENDIA_PD + {m}_KERENDIA_RV").count()
    print(f"{m}: rows where OVERALL != PD+RJ+RV: {bad_sum} | rows where APPROVED != PD+RV: {bad_app}")
    assert bad_sum == 0 and bad_app == 0, "Unexpected CLAIM_TYPE values or split error"
assert summary.collect()[0]["last_month"] >= lookback_end_max, "Claims do not reach the latest cohort lookback end"

In [0]:
# Lifetime totals per HCP from the monthly table vs the saved HCP-level table of Kerendia_NBRX_From_Raw_FCT_RX.
# The saved table was pulled 2026-10-16 (claims to 2026-09-18), so only months up to 2026-08 are compared.
ref = spark.read.format("delta").load(HCP_LEVEL_REF_PATH)
mine = (monthly_all.filter(F.col("SVC_DT_M") <= "2026-08-01").groupBy("PROVIDER_ID")
        .agg(F.sum("NBRX_KERENDIA_OVERALL").alias("nbrx_all"), F.sum("TRX_KERENDIA_OVERALL").alias("trx_all")))
ref_tot = ref.agg(F.sum("NBRX_KERENDIA_OVERALL").alias("ref_nbrx_all"), F.sum("TRX_KERENDIA_OVERALL").alias("ref_trx_all")).collect()[0]
mine_tot = mine.agg(F.sum("nbrx_all").alias("nbrx_all"), F.sum("trx_all").alias("trx_all")).collect()[0]
print(f"NBRx all claim types: monthly table (to 2026-08) {mine_tot['nbrx_all']:,} | saved HCP table (to 2026-09-18) {ref_tot['ref_nbrx_all']:,}")
print(f"TRx  all claim types: monthly table (to 2026-08) {mine_tot['trx_all']:,} | saved HCP table (to 2026-09-18) {ref_tot['ref_trx_all']:,}")

In [0]:
max_n = builtins.max(WINDOWS)
monthly = monthly_all.join(F.broadcast(hcp_universe), "PROVIDER_ID", "left_semi")

# Step 5: months from add_months(lookback_end, -(N-1)) to lookback_end, per cohort
joined = (monthly.join(F.broadcast(coh),
                       (F.col("SVC_DT_M") <= F.col("LOOKBACK_END_M")) &
                       (F.col("SVC_DT_M") >= F.add_months(F.col("LOOKBACK_END_M"), -(max_n - 1))), "inner")
          .withColumn("MONTHS_BACK", F.months_between("LOOKBACK_END_M", "SVC_DT_M").cast("int")))

def feat_name(metric, infix, n):
    return f"KERENDIA_{metric}{infix}_IN_LAST_{n}_MONTH"

# (feature column, monthly source column) for every metric x claim-type group x window
feature_map = [(feat_name(m, infix, n), f"{m}_KERENDIA_{grp}", n)
               for m in ("NBRX", "TRX") for grp, infix in FEATURE_GROUPS.items() for n in WINDOWS]
feature_cols = [f for f, _, _ in feature_map]
aggs = [F.sum(F.when(F.col("MONTHS_BACK") < n, F.col(src)).otherwise(0)).alias(f) for f, src, n in feature_map]
windows = joined.groupBy("PROVIDER_ID", "cohort_id").agg(*aggs)
print(f"{len(feature_cols)} features")

# Step 6: every universe HCP x every claims cohort; 0 when no Kerendia claim in the window
kerendia_prior_writing = (hcp_universe.crossJoin(F.broadcast(coh))
    .join(windows, ["PROVIDER_ID", "cohort_id"], "left")
    .fillna(0, subset=feature_cols)
    .withColumnRenamed("LOOKBACK_END_M", "SVC_DT_M")
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "cohort_id"))
    .withColumn("PULL_DT", F.lit(dt.today().strftime("%Y_%m_%d")))
    .select("PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID", *[F.col(c).cast("long").alias(c) for c in feature_cols], "PULL_DT"))

In [0]:
kpw = kerendia_prior_writing
n_rows = kpw.count()
assert n_rows == n_hcp * len(cohorts), (n_rows, n_hcp * len(cohorts))
assert kpw.select("HCP_COHORT_ID").distinct().count() == n_rows, "HCP_COHORT_ID not unique"
assert kpw.join(hcp_universe, "PROVIDER_ID", "left_anti").count() == 0
neg = kpw.filter(" OR ".join(f"{c} < 0" for c in feature_cols)).count()
non_mono = kpw.filter(" OR ".join(f"{feat_name(m, i, a)} > {feat_name(m, i, b)}"
                                  for m in ("NBRX", "TRX") for i in FEATURE_GROUPS.values() for a, b in zip(WINDOWS, WINDOWS[1:]))).count()
nbrx_gt_trx = kpw.filter(" OR ".join(f"{feat_name('NBRX', i, n)} > {feat_name('TRX', i, n)}"
                                     for i in FEATURE_GROUPS.values() for n in WINDOWS)).count()
# Groups must add up: overall = approved + rejected (OVERALL = PD + RJ + RV, APPROVED = PD + RV)
not_additive = kpw.filter(" OR ".join(f"{feat_name(m, '', n)} != {feat_name(m, '_APPROVED', n)} + {feat_name(m, '_REJECTED', n)}"
                                      for m in ("NBRX", "TRX") for n in WINDOWS)).count()
sub_gt_app = kpw.filter(" OR ".join(f"{feat_name(m, '_REVERSED', n)} > {feat_name(m, '_APPROVED', n)}"
                                    for m in ("NBRX", "TRX") for n in WINDOWS)).count()
print(f"rows {n_rows:,} | negatives {neg} | longer window < shorter window {non_mono} | NBRx > TRx {nbrx_gt_trx} "
      f"| overall != approved + rejected {not_additive} | reversed > approved {sub_gt_app}")
assert neg == 0 and non_mono == 0 and nbrx_gt_trx == 0 and not_additive == 0 and sub_gt_app == 0

# Independent check (3.0 method) for the latest cohort: full calendar, empty months = 0, rolling sum over exactly N rows
c_last = cohorts.iloc[-1]
end = pd.to_datetime(c_last.lookback_end_date).date()
start = (pd.Timestamp(end) - pd.DateOffset(months=max_n + 5)).date()
cal = (monthly.select("PROVIDER_ID").distinct()
       .crossJoin(spark.sql(f"SELECT explode(sequence(DATE'{start}', DATE'{end}', interval 1 month)) AS SVC_DT_M")))
src_cols = sorted({src for _, src, _ in feature_map})
m = (cal.join(monthly.select("PROVIDER_ID", "SVC_DT_M", *src_cols), ["PROVIDER_ID", "SVC_DT_M"], "left")
       .fillna(0, subset=src_cols))
w = Window.partitionBy("PROVIDER_ID").orderBy("SVC_DT_M")
m = m.select("PROVIDER_ID", "SVC_DT_M", *[F.sum(src).over(w.rowsBetween(-n + 1, 0)).alias(f"chk_{f}") for f, src, n in feature_map])
chk = m.filter(F.col("SVC_DT_M") == F.lit(end)).drop("SVC_DT_M")
cmp = kpw.filter(F.col("cohort_id") == c_last.cohort_id).join(chk, "PROVIDER_ID", "left").fillna(0)
mismatch = cmp.filter(" OR ".join(f"{f} != chk_{f}" for f in feature_cols)).count()
print(f"{c_last.cohort_id}: mismatches vs rolling-sum method = {mismatch}")
assert mismatch == 0

# Context: paid-only NBRx from raw claims vs LRx V6 (paid) Kerendia NBRx, universe HCPs, latest cohort month
lrx = spark.sql(f"""SELECT PROVIDER_ID, SUM(NBRX) AS NBRX FROM {LRX_TABLE}
                    WHERE UPPER(PRODUCT_GROUP) = 'KERENDIA' AND trunc(PERIOD_ID, 'month') = DATE'{end}' GROUP BY 1""")
lrx_tot = lrx.join(hcp_universe, "PROVIDER_ID", "left_semi").agg(F.sum("NBRX")).collect()[0][0]
raw = monthly.filter(F.col("SVC_DT_M") == F.lit(end)).agg(F.sum("NBRX_KERENDIA_PD"), F.sum("NBRX_KERENDIA_OVERALL")).collect()[0]
print(f"{end}: raw paid NBRx {raw[0]:,} | raw all-claims NBRx {raw[1]:,} | LRx V6 Kerendia NBRx {lrx_tot:,}")

In [0]:
display(kpw.groupBy("cohort_id", "SVC_DT_M")
        .agg(F.sum(F.when(F.col("KERENDIA_NBRX_IN_LAST_1_MONTH") > 0, 1).otherwise(0)).alias("hcps_nbrx_1m"),
             F.sum(F.when(F.col("KERENDIA_NBRX_IN_LAST_6_MONTH") > 0, 1).otherwise(0)).alias("hcps_nbrx_6m"),
             *[F.sum(feat_name("NBRX", i, 1)).alias(f"nbrx{i.lower() or '_overall'}_1m") for i in FEATURE_GROUPS.values()],
             *[F.sum(feat_name("TRX", i, 1)).alias(f"trx{i.lower() or '_overall'}_1m") for i in FEATURE_GROUPS.values()])
        .orderBy("SVC_DT_M"))

In [0]:
os.makedirs(HCP_FEATURES_OUT_31, exist_ok=True)
(kerendia_prior_writing.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(OUTPUT_PATH))
saved = spark.read.format("delta").load(OUTPUT_PATH)
print(f"Saved {saved.count():,} rows, {len(saved.columns)} columns -> {OUTPUT_PATH}")